# Stage A — ControlNet training (DPED iPhone → DSLR)

Kaggle setup:
1. Add-ons → Secrets: `KAGGLE_USERNAME` + `KAGGLE_KEY` (for kagglehub DPED download)
2. Accelerator: **GPU T4 x2** (or P100)
3. Internet: **On** (needed for pip + model downloads)
4. Persist files across sessions via outputs: checkpoints land in `/kaggle/working/runs/stage_a`

In [ ]:
# Cell 1: install deps
!pip -q install 'torch>=2.1,<2.6' diffusers==0.31.0 transformers==4.44.2 accelerate safetensors bitsandbytes scipy kagglehub tqdm pillow
import torch
print('CUDA:', torch.cuda.is_available(), '| device count:', torch.cuda.device_count())

In [ ]:
# Cell 2: get project code
# ROUTE 1 (easiest): Kaggle.com -> Datasets -> New Dataset -> upload dped-ldm-upload.zip (Kaggle auto-extracts),
#   then in this notebook: right panel -> Add Input -> Your Datasets -> attach it. REPO_URL stays empty.
# ROUTE 2: put code on GitHub and paste the URL below.
import os, glob, shutil, subprocess
REPO_URL = ""  # e.g. "https://github.com/YOUR_USER/dped-ldm.git" (Route 2 only)
W = '/kaggle/working'
os.chdir(W)
if not os.path.exists(f'{W}/src/common.py'):
    hits = glob.glob('/kaggle/input/*/src/common.py')
    if hits:
        root = os.path.dirname(os.path.dirname(hits[0]))
        for item in ('src', 'tests', 'requirements.txt', 'README.md'):
            s, d = os.path.join(root, item), os.path.join(W, item)
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            elif os.path.exists(s):
                shutil.copy2(s, d)
        print('Code loaded from attached Kaggle dataset.')
    elif REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, f'{W}/repo'], check=True)
        shutil.copytree(f'{W}/repo/src', f'{W}/src', dirs_exist_ok=True)
        print('Code cloned from GitHub.')
    else:
        raise RuntimeError('Code not found: attach code dataset (Route 1) or set REPO_URL (Route 2)')
sys.path.insert(0, f'{W}/src')
print('src files:', sorted(os.listdir(f'{W}/src')))

In [ ]:
# Cell 3: data prep (download + crops + latent cache). ~30-45 min on first run.
import sys
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ.setdefault('KAGGLE_USERNAME', _s.get_secret('KAGGLE_USERNAME'))
    os.environ.setdefault('KAGGLE_KEY', _s.get_secret('KAGGLE_KEY'))
    print('Kaggle credentials loaded from notebook Secrets.')
except Exception as _e:
    print('Secrets unavailable (kagglehub may auto-auth on Kaggle):', _e)
sys.path.insert(0, '/kaggle/working/src')
import argparse
from pathlib import Path
import prepare_data

sys.argv = ['prepare_data.py', '--source', 'auto', '--out', 'data/processed/iphone']
try:
    prepare_data.main()
except SystemExit:
    pass

In [ ]:
# Cell 4: build latent cache if not already built by prepare_data
import sys; sys.path.insert(0, '/kaggle/working/src')
import torch
from pathlib import Path
from common import load_vae, build_latent_cache, PairedCropDataset, LatentPairDataset

device = torch.device('cuda')
cache_dir = Path('data/processed/iphone/latent_cache')
try:
    LatentPairDataset(cache_dir, 'train'); print('cache exists')
except (FileNotFoundError, AssertionError):
    vae = load_vae('fp16', device)
    for split in ('train', 'val'):
        d = Path('data/processed/iphone/crops') / split
        build_latent_cache(vae, PairedCropDataset(d/'phone', d/'dslr'), cache_dir, split, 8, device)


In [ ]:
# Cell 5: Stage A training. ~10-12h for 15k steps at batch 16 on T4 x2 (single-GPU script).
# Session chain: set --resume and re-run; checkpoints auto-save to runs/stage_a every 2000 steps.
import sys; sys.path.insert(0, '/kaggle/working/src')
sys.argv = ['train_controlnet.py',
            '--data', 'data/processed/iphone',
            '--out', 'runs/stage_a',
            '--steps', '15000', '--batch', '16',
            '--resume']
import train_controlnet
train_controlnet.main()

In [ ]:
# Cell 6 (optional, same session): quick val grids + metrics on a recent checkpoint
import sys, glob; sys.path.insert(0, '/kaggle/working/src')
ckpts = sorted(glob.glob('runs/stage_a/controlnet_step_*.pt'))
if ckpts:
    sys.argv = ['infer.py', '--ckpt', ckpts[-1], '--mode', 'crops', '--num', '8']
    import infer; infer.main()
    sys.argv = ['metrics.py', '--gt', 'data/processed/iphone/crops/val/dslr',
                '--pred', 'runs/infer', '--max-images', '50']
    import metrics; metrics.main()